# Cluster caps for the direct[1h] penalty method

The direct[1h] cap needs a price feed, which most tokens traded on CoW do not have. The agreed production ladder has four rungs: correlated pairs (by CoW's own fee classification) get a rule based cap; pairs with data, real or synthetic, get direct[1h]; pairs whose data is too thin for the quantile fall back to their group's median; pairs with no data start at a high default and graduate as data accrues. This notebook calibrates those group caps and defaults: cluster Binance spot assets by their volatility behaviour, compute direct[1h] caps for members with data, and read off the cluster percentile caps. The target revert rate is 8 percent throughout, matching the penalty notebooks.

Design in brief. All active USDT spot pairs are clustered from two months of 1 minute klines, two ways: dynamic time warping on daily volatility paths (operative) and Wasserstein clustering on the return distributions (check). Exact caps come from 1 second klines for a representative subset, validated against the tick pipeline. Correlated and synthetic pairs (wstETH proxy, COW/ETH, BABY/KAITO) are built by dividing the legs' price series, per the agreed synthetic construction. Clusters are fit on the first month and every revert rate is evaluated on the second month, with leave one out cluster caps.

In [ ]:
import os, io, json, time, zipfile, warnings, urllib.request
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")

T_EXCL = 26                     # exclusivity window (s)
Q_TARGET = 0.08                 # target revert rate (8 percent, comparable with the penalty notebooks)
W_1H = 3600 // T_EXCL           # trailing-hour window count = 138
CLUSTER_PCTL = 75               # cluster cap percentile
PCTL_GRID = [50, 75, 90]
DATE_START, DATE_END = "2026-05-22", "2026-07-21"
DAYS = [d.strftime("%Y-%m-%d") for d in pd.date_range(DATE_START, DATE_END)]
H1_END = "2026-06-21"           # clusters fit on DATE_START..H1_END
DAYS_H1 = [d for d in DAYS if d <= H1_END]
DAYS_H2 = [d for d in DAYS if d > H1_END]
K_RANGE = range(3, 9)
N_BUDGET = 400                  # symbols with exact 1s-kline caps
MIN_CLUSTER = 10                # smallest admissible cluster in the k scan

ROOT = os.path.expanduser("~/dev/penalty-research/data")
CACHE_1M = os.path.join(ROOT, "binance_klines/1m")
CACHE_1S = os.path.join(ROOT, "binance_klines/1s")
CACHE_NPY = os.path.join(ROOT, "binance_klines/npy")
CACHE_TICKS = os.path.join(ROOT, "binance_ticks")
for p in (CACHE_1M, CACHE_1S, CACHE_NPY): os.makedirs(p, exist_ok=True)

pio.renderers.default = "png"
pio.renderers["png"].scale = 2
GRAY, INK, GRID, SURF, AXIS = "#898781", "#0b0b0b", "#e1e0d9", "#fcfcfb", "#c3c2b7"
C_CLUSTER = ["#2a78d6", "#1baf7a", "#eb6834", "#4a3aa7", "#c22f2f", "#c9a227", "#d02f8e", "#7a7f2a"]
CAP_ACC = "#e0a63a"
C_COR = "#3d7a78"                # reserved color for the rule-based correlated group

def style(fig, w=1050, h=430, title=None):
    fig.update_layout(width=w, height=h, template="none", paper_bgcolor=SURF, plot_bgcolor=SURF,
                      font=dict(size=13, color=INK), margin=dict(l=65, r=30, t=60, b=50),
                      legend=dict(bgcolor="rgba(0,0,0,0)"))
    if title: fig.update_layout(title=dict(text=title, font=dict(size=15)))
    fig.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=AXIS)
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=AXIS)
    return fig
rng = np.random.default_rng(7)
print(f"window {T_EXCL}s | target {Q_TARGET:.0%} | {DAYS[0]} .. {DAYS[-1]} "
      f"({len(DAYS)} days, fit month ends {H1_END})")

## 1. Universe

All symbols currently trading on Binance spot, restricted to USDT quotes. One pair per base asset avoids the same asset entering twice under different quotes, USDT books are the deepest so features are least distorted by microstructure, and caps are in bps of order size so the common quote makes them directly comparable. Cross rates enter through the correlated layer and the synthetic feeds of Section 2b; a production mapping assigns any non USDT pair through its base asset or its own synthetic series.

Two groups leave the analysis entirely: leveraged tokens, which are products rather than assets, and symbols younger than 45 days or with almost no trading, which cannot support feature estimates and are exactly the population the highest cap default exists for. USD pegged stable pairs stay in; the correlated layer of Section 2b claims them by rule before any clustering happens, tick floors and all.

In [ ]:
EXINFO = os.path.join(ROOT, "binance_klines/exchangeInfo_2026-07-22.json")
if not os.path.exists(EXINFO):
    with urllib.request.urlopen("https://api.binance.com/api/v3/exchangeInfo", timeout=60) as r:
        open(EXINFO, "wb").write(r.read())
info = json.load(open(EXINFO))["symbols"]

rows = []
for s in info:
    if s["status"] != "TRADING" or s["quoteAsset"] != "USDT": continue
    if not s["symbol"].isascii(): continue      # meme listings with CJK names break URL encoding
    if not s.get("isSpotTradingAllowed", True): continue
    tick = [float(f["tickSize"]) for f in s["filters"] if f["filterType"] == "PRICE_FILTER"]
    rows.append((s["symbol"], s["baseAsset"], tick[0] if tick else np.nan))
TICK_ALL = {s2["symbol"]: float(f["tickSize"]) for s2 in info if s2["status"] == "TRADING"
            for f in s2["filters"] if f["filterType"] == "PRICE_FILTER"}
uni = pd.DataFrame(rows, columns=["symbol", "base", "tickSize"]).set_index("symbol")
n0 = len(uni)
bases = set(uni.base)
lev = uni.base.map(lambda b: (b.endswith("UP") and b[:-2] in bases)
                   or (b.endswith("DOWN") and b[:-4] in bases)
                   or (b.endswith("BULL") and b[:-4] in bases)
                   or (b.endswith("BEAR") and b[:-4] in bases))
uni = uni[~lev]
print(f"TRADING x USDT spot: {n0} | after leveraged filter: {len(uni)}")
print("USD-pegged stable pairs stay in the universe; the COR classification routes them "
      "to the rule-based correlated group before any clustering")

## 2. Kline data

1 minute klines for the whole universe over the two months: monthly zips for May and June, daily zips for July, from data.binance.vision. Files are cached with atomic writes; a 404 means the symbol simply has no data for that period and is recorded, not retried. Each symbol is immediately reduced to compact daily summaries so the 35 million raw rows never sit in memory together.

In [ ]:
def dl(url, path):
    if os.path.exists(path) or os.path.exists(path + ".404"): return
    for att in range(4):
        try:
            with urllib.request.urlopen(url, timeout=45) as r:
                data = r.read()
            open(path + ".part", "wb").write(data)
            os.replace(path + ".part", path)
            return
        except urllib.error.HTTPError as e:
            if e.code == 404: open(path + ".404", "w").close()
            return
        except Exception:
            time.sleep(2*(att + 1))     # transient network error: retry, else leave missing

def files_1m(sym):
    out = [(f"https://data.binance.vision/data/spot/monthly/klines/{sym}/1m/{sym}-1m-2026-{mm}.zip",
            os.path.join(CACHE_1M, f"{sym}-1m-2026-{mm}.zip")) for mm in ("05", "06")]
    out += [(f"https://data.binance.vision/data/spot/daily/klines/{sym}/1m/{sym}-1m-2026-07-{dd:02d}.zip",
             os.path.join(CACHE_1M, f"{sym}-1m-2026-07-{dd:02d}.zip")) for dd in range(1, 22)]
    return out

jobs = [uj for sym in list(uni.index) + ["WBETHETH"] for uj in files_1m(sym)]
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    for i, _ in enumerate(ex.map(lambda a: dl(*a), jobs)):
        if i % 2000 == 0: print(f"  {i}/{len(jobs)} files, {time.time() - t0:.0f}s", flush=True)
print(f"1m downloads done in {time.time() - t0:.0f}s")

In [ ]:
SUMM_PKL = os.path.join(ROOT, "binance_klines/summ_1m_v2.pkl")
QSK_GRID = np.linspace(0.005, 0.995, 129)   # trimmed quantile grid for distribution sketches
def parse_sym(sym):
    frames = []
    for _, path in files_1m(sym):
        if not os.path.exists(path): continue
        try:
            with zipfile.ZipFile(path) as z:
                name = z.namelist()[0]
                hh = not z.open(name).read(16)[:1].isdigit()
                df = pd.read_csv(z.open(name), header=0 if hh else None,
                                 usecols=[0, 4, 7], names=["ts", "close", "qvol"])
        except Exception:
            continue
        frames.append(df)
    if not frames: return None
    df = pd.concat(frames)
    unit = "us" if df["ts"].iloc[0] > 10**14 else "ms"
    df["day"] = pd.to_datetime(df["ts"], unit=unit, utc=True).dt.strftime("%Y-%m-%d")
    df = df[(df.day >= DATE_START) & (df.day <= DATE_END)].sort_values("ts")
    if len(df) < 1000: return None
    r = df.close.pct_change().to_numpy()[1:]
    day = df.day.to_numpy()[1:]
    g = {}
    for h, dsel in (("h1", DAYS_H1), ("h2", DAYS_H2)):
        m = np.isin(day, dsel)
        if m.sum() < 500: g[h] = None; continue
        rm = r[m]
        g[h] = dict(qt=abs(float(np.quantile(rm, Q_TARGET)))*1e4,
                    rv=float(np.sqrt(np.mean(rm**2)))*1e4,
                    zero=float(np.mean(rm == 0.0)),
                    qsk=(np.quantile(rm, QSK_GRID)*1e4).astype(np.float32))
    dd = pd.DataFrame({"r2": r**2, "z": r == 0.0, "day": day}).groupby("day")
    vold = (dd.r2.mean()**0.5)*1e4
    qv = df.groupby("day").qvol.sum()
    return dict(days=list(vold.index), vol_daily=vold.to_numpy(),
                qvol_med=float(qv.median()), med_close=float(df.close.median()),
                n_days=len(vold), first=str(df.day.iloc[0]), h1=g["h1"], h2=g["h2"])

if os.path.exists(SUMM_PKL):
    SUMM = pd.read_pickle(SUMM_PKL)
else:
    SUMM, t0 = {}, time.time()
    for i, sym in enumerate(uni.index):
        s = parse_sym(sym)
        if s is not None: SUMM[sym] = s
        if i % 50 == 0: print(f"  parsed {i}/{len(uni)}, {time.time() - t0:.0f}s", flush=True)
    pd.to_pickle(SUMM, SUMM_PKL)
    print(f"parse done in {time.time() - t0:.0f}s")

RECENT = sorted(s for s in SUMM if SUMM[s]["first"] > DATE_START or SUMM[s]["n_days"] < 45)
ILLIQ = sorted(s for s in SUMM if s not in RECENT
               and (SUMM[s]["h1"] or {}).get("zero", 1) > 0.8)      # only near-dead pairs excluded
CLUST = sorted(s for s in SUMM if s not in RECENT and s not in ILLIQ and SUMM[s]["h1"])
print(f"symbols with data: {len(SUMM)} | clustered: {len(CLUST)} | "
      f"young (default bucket test set): {len(RECENT)} | too illiquid: {len(ILLIQ)}")

## 2b. The correlated layer and the synthetic feeds

Before any statistical clustering, pairs that CoW itself treats as correlated form their own rule based group, COR. The classification is CoW's own: the CMS correlated token buckets behind the reduced volume fee (a pair is correlated when both tokens sit in one bucket). The DTW and distribution clustering below then run only on the uncorrelated remainder.

The correlated and synthetic series enter the analysis as regular feeds: WBETHETH from its own listing, and the synthetics as the ratio of their legs' 1 minute closes, pushed through the identical summary code as every other pair. The synthetic therefore carries its own inflated short horizon volatility into the features, deliberately: that is the feed the production estimator would actually see for a pair with no listing.

In [ ]:
CMS_JSON = os.path.join(ROOT, "binance_klines/cow_correlated_tokens.json")
if not os.path.exists(CMS_JSON):
    url = ("https://cms.cow.finance/api/correlated-tokens?fields%5B%5D=tokens"
           "&populate%5Bnetwork%5D%5Bfields%5D%5B%5D=chainId&pagination%5BpageSize%5D=100")
    with urllib.request.urlopen(url, timeout=60) as r:
        open(CMS_JSON, "wb").write(r.read())
cms = json.load(open(CMS_JSON))["data"]
eth_buckets = [ {v.upper() for v in e["attributes"]["tokens"].values()}
                for e in cms
                if e["attributes"]["network"]["data"]["attributes"]["chainId"] == 1 ]
b_stable, b_eth = max(eth_buckets, key=len), min(eth_buckets, key=len)
ALIAS_STABLE = {"USDT", "USDC", "EUR", "EURI", "AEUR", "RLUSD", "FDUSD", "TUSD", "USD1", "XUSD", "USDP", "DAI", "PYUSD", "USDE", "BUSD"}
STABLE_SYMS = b_stable | ALIAS_STABLE
ETH_SYMS = b_eth | {"ETH", "WBETH"}

SPECIAL_LEGS = {"WBETHETH": ("WBETH", "ETH"), "WBETH/ETH syn": ("WBETH", "ETH"),
                "COW/ETH syn": ("COW", "ETH"), "BABY/KAITO syn": ("BABY", "KAITO")}
def cow_class(name):
    a, b = SPECIAL_LEGS.get(name, (name[:-4], "USDT"))
    a, b = a.upper(), b.upper()
    if (a in STABLE_SYMS and b in STABLE_SYMS) or (a in ETH_SYMS and b in ETH_SYMS):
        return "correlated"
    return "standard"

def load_1m_closes(sym):
    frames = []
    for _, path in files_1m(sym):
        if not os.path.exists(path): continue
        try:
            with zipfile.ZipFile(path) as z:
                name = z.namelist()[0]
                hh = not z.open(name).read(16)[:1].isdigit()
                frames.append(pd.read_csv(z.open(name), header=0 if hh else None,
                                          usecols=[0, 4], names=["ts", "close"]))
        except Exception: continue
    df = pd.concat(frames)
    unit = "us" if df["ts"].iloc[0] > 10**14 else "ms"
    ser = pd.Series(df["close"].to_numpy(),
                    index=pd.to_datetime(df["ts"], unit=unit)).sort_index()
    return ser[(ser.index >= DATE_START) & (ser.index < pd.Timestamp(DATE_END) + pd.Timedelta(days=1))]

def summ_from_series(px):
    # identical math to parse_sym, applied to a 1m close series
    r = px.pct_change().to_numpy()[1:]
    day = px.index.strftime("%Y-%m-%d").to_numpy()[1:]
    g = {}
    for h, dsel in (("h1", DAYS_H1), ("h2", DAYS_H2)):
        m = np.isin(day, dsel)
        if m.sum() < 500: g[h] = None; continue
        rm = r[m]
        g[h] = dict(qt=abs(float(np.quantile(rm, Q_TARGET)))*1e4,
                    rv=float(np.sqrt(np.mean(rm**2)))*1e4,
                    zero=float(np.mean(rm == 0.0)),
                    qsk=(np.quantile(rm, QSK_GRID)*1e4).astype(np.float32))
    dd = pd.DataFrame({"r2": r**2, "day": day}).groupby("day")
    vold = (dd.r2.mean()**0.5)*1e4
    return dict(days=list(vold.index), vol_daily=vold.to_numpy(), qvol_med=np.nan,
                med_close=float(px.median()), n_days=len(vold), first=str(day[0]),
                h1=g["h1"], h2=g["h2"])

CLOSE_1M = {sym: load_1m_closes(sym) for sym in
            ("WBETHETH", "WBETHUSDT", "ETHUSDT", "COWUSDT", "BABYUSDT", "KAITOUSDT")}
def ratio_1m(a, b):
    idx = CLOSE_1M[a].index.union(CLOSE_1M[b].index)
    return (CLOSE_1M[a].reindex(idx).ffill()/CLOSE_1M[b].reindex(idx).ffill()).dropna()
CLOSE_1M["WBETH/ETH syn"] = ratio_1m("WBETHUSDT", "ETHUSDT")
CLOSE_1M["COW/ETH syn"] = ratio_1m("COWUSDT", "ETHUSDT")
CLOSE_1M["BABY/KAITO syn"] = ratio_1m("BABYUSDT", "KAITOUSDT")

LEG_PAIRS = {"WBETH/ETH syn": ("WBETHUSDT", "ETHUSDT"), "COW/ETH syn": ("COWUSDT", "ETHUSDT"),
             "BABY/KAITO syn": ("BABYUSDT", "KAITOUSDT")}
TICK_BPS = {}
for name in ("WBETHETH",) + tuple(LEG_PAIRS):
    summ_n = summ_from_series(CLOSE_1M[name])
    if name == "WBETHETH":
        summ_n["qvol_med"] = SUMM.get("ETHUSDT", {}).get("qvol_med", 1e6)
        TICK_BPS[name] = TICK_ALL["WBETHETH"]/summ_n["med_close"]*1e4
    else:
        la, lb = LEG_PAIRS[name]
        summ_n["qvol_med"] = min(SUMM[la]["qvol_med"], SUMM[lb]["qvol_med"])
        TICK_BPS[name] = max(TICK_ALL[l]/SUMM[l]["med_close"]*1e4 for l in (la, lb))
    SUMM[name] = summ_n

COR = sorted({t for t in SUMM if t not in RECENT and cow_class(t) == "correlated"})
CLUST_UNC = sorted([t for t in CLUST if cow_class(t) != "correlated"]
                   + ["COW/ETH syn", "BABY/KAITO syn"])
print(f"CoW buckets (mainnet): stables/RWAs {len(b_stable)}, ETH-correlated {len(b_eth)}; "
      "rule: both tokens in one bucket = correlated")
print(f"COR layer ({len(COR)}): " + ", ".join(t.replace("USDT", "") for t in COR))
print(f"clustering population (uncorrelated incl. synthetics): {len(CLUST_UNC)}")

## 3. How much of CoW flow can this reach

Before clustering anything it is worth checking what share of actual CoW mainnet activity involves assets a Binance based cap can serve at all. The realized reverts extract gives attempts per asset per day on Ethereum; each asset is matched to a Binance USDT pair directly or through a stated proxy (WETH through ETH, WBTC through BTC, wstETH through ETH). Attempt counts are a proxy for order flow, not notional, and this is mainnet only.

In [ ]:
cow = pd.read_csv("ethereum_daily_reverts_per_asset.csv")
cow = cow[cow.asset != "_ALL"]
att = cow.groupby("asset").n_attempts.sum().sort_values(ascending=False)
ALIAS = {"WETH": "ETH", "ETH": "ETH", "WBTC": "BTC", "CBBTC": "BTC", "TBTC": "BTC",
         "WSTETH": "ETH", "STETH": "ETH", "RETH": "ETH", "WEETH": "ETH", "CBETH": "ETH",
         "USDC": "_STABLE", "USDT": "_STABLE", "DAI": "_STABLE", "USDS": "_STABLE",
         "USDE": "_STABLE", "FDUSD": "_STABLE", "PYUSD": "_STABLE", "GHO": "_STABLE",
         "LUSD": "_STABLE", "FRAX": "_STABLE"}
def cov(a):
    a = str(a).upper()
    if "STABLE" in a: return "stable (pegged rule)"
    b = ALIAS.get(a, a)
    if b == "_STABLE": return "stable (pegged rule)"
    if b + "USDT" in SUMM: return "direct" if ALIAS.get(a, a) == a else "proxy"
    return "uncovered"
cw = att.groupby(att.index.map(cov)).sum()
tot = att.sum()
print("share of CoW mainnet attempts by coverage:")
for kk in ("direct", "proxy", "stable (pegged rule)", "uncovered"):
    if kk in cw: print(f"  {kk:22s} {cw[kk]/tot:6.1%}")
unc = att[att.index.map(cov) == "uncovered"]
print("top uncovered assets by attempts:")
print(unc.head(12).to_string())

## 4. Two ways of clustering volatility behaviour

The clustering only has to answer one calibration question: which pairs behave alike enough to share a cap level. Two methods, both operating on the volatility behaviour itself rather than on hand picked summary numbers.

Approach A, the operative one, clusters the daily volatility paths with k means under dynamic time warping. Each pair is its 31 day series of daily realized volatility over the fit month, standardized with one global mean and scale (not per series, since the volatility level is the primary cap determinant and per series scaling would erase it). Two pairs co-cluster when their volatility level and its evolution through the month align.

Approach B clusters the return distributions themselves, using the Wasserstein distance, an established approach for financial return data (Horvath, Issa and Muguruza's market regime clustering is the reference application). For one dimensional distributions the Wasserstein-2 distance equals the L2 distance between quantile functions, so k means on each pair's quantile sketch of 1 minute returns is exact Wasserstein clustering. This sees the whole shape of the distribution: level, spread, fat tails, and the mass at zero that tick quantization creates. Simpler alternatives compress that shape into one number (standard deviation, or the coefficient of variation of volatility, which is included in the feature table above); the quantile sketch contains what all of those measure and is no harder to compute, which is why it fills this slot. Agreement between A and B is evidence the partition reflects the data, not the algorithm.

In [ ]:
from statistics import NormalDist
Z_TAIL = abs(NormalDist().inv_cdf(Q_TARGET))     # Gaussian z at the target quantile

def feat_row(sym, half):
    s = SUMM[sym]; g = s[half]
    if g is None: return None
    dsel = DAYS_H1 if half == "h1" else DAYS_H2
    m = np.isin(s["days"], dsel)
    if m.sum() < 15: return None
    v = s["vol_daily"][m]
    v = v[np.isfinite(v) & (v > 0)]
    if len(v) < 10: return None
    return dict(volT=np.median(v)*np.sqrt(T_EXCL/60),
                vol_of_vol=float(np.std(v)),
                cv_vol=float(np.std(v)/np.mean(v)),
                q_tail=g["qt"]*np.sqrt(T_EXCL/60),
                tail_ratio=g["qt"]/(Z_TAIL*g["rv"] + 1e-9),
                tick_bps=TICK_BPS.get(sym, TICK_ALL.get(sym, np.nan)/SUMM[sym]["med_close"]*1e4),
                zero_share=g["zero"],
                log_qvol=np.log10(max(s["qvol_med"], 1.0)))

POP = sorted(set(CLUST_UNC) | set(COR))
F1 = pd.DataFrame({s: r for s in POP if (r := feat_row(s, "h1"))}).T
F2 = pd.DataFrame({s: r for s in POP if (r := feat_row(s, "h2"))}).T
print(f"descriptive features: {F1.shape[0]} symbols x {F1.shape[1]} (fit month; raw units, "
      "used for tick pinning, subset strata and cluster profiles, not fed to any clusterer)")
display(F1.describe().loc[["25%", "50%", "75%"]].round(2))

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from tslearn.clustering import TimeSeriesKMeans
from tslearn.metrics import cdist_dtw

def vol_series(dsel):
    # tolerate up to 2 missing days per symbol (late-published daily files), forward-filled
    sers, syms = [], []
    for sym in CLUST_UNC:
        ss = SUMM[sym]
        pos = {d: i for i, d in enumerate(ss["days"])}
        v = np.array([ss["vol_daily"][pos[d]] if d in pos else np.nan for d in dsel])
        v[~(v > 0)] = np.nan
        if np.isnan(v).sum() <= 2:
            sers.append(pd.Series(v).ffill().bfill().to_numpy()); syms.append(sym)
    S = np.array(sers)
    S = (S - S.mean())/S.std()
    S = np.clip(S, None, float(np.quantile(S, 0.995)))   # tame micro-price outliers (BTTC-class)
    return S, syms

SER, syms_a = vol_series(DAYS_H1)
import hashlib
_dkey = hashlib.md5(("v2clip" + ",".join(sorted(syms_a)) + H1_END + str(list(K_RANGE))).encode()).hexdigest()[:12]
DTW_CACHE = os.path.join(CACHE_NPY, f"dtw_scan_{_dkey}.pkl")
if os.path.exists(DTW_CACHE):
    sc, K_A, _lblraw = pd.read_pickle(DTW_CACHE)
    print("(DTW scan loaded from cache)")
else:
    D = cdist_dtw(SER)
    scan = []
    for k in K_RANGE:
        km = TimeSeriesKMeans(n_clusters=k, metric="dtw", random_state=7, n_init=2, max_iter=30).fit(SER)
        sizes = np.bincount(km.labels_)
        scan.append((k, silhouette_score(D, km.labels_, metric="precomputed"), sizes.min()))
    sc = pd.DataFrame(scan, columns=["k", "silhouette", "min_size"]).set_index("k")
    ok = sc[sc.min_size >= MIN_CLUSTER]
    K_A = int((ok if len(ok) else sc).silhouette.idxmax())
    _lblraw = None
print(sc.round(3).to_string(), f"\nchosen k = {K_A} (max DTW silhouette with every cluster >= {MIN_CLUSTER})")

if _lblraw is not None:
    raw_lbl = _lblraw.copy()
else:
    KM_A = TimeSeriesKMeans(n_clusters=K_A, metric="dtw", random_state=7, n_init=3, max_iter=50).fit(SER)
    raw_lbl = pd.Series(KM_A.labels_, index=syms_a)
    pd.to_pickle((sc, K_A, raw_lbl), DTW_CACHE)
# fold clusters below the minimum size into their nearest neighbour by volatility level:
# a lone micro-price outlier must not occupy a cluster of its own
meds = raw_lbl.groupby(raw_lbl).apply(
    lambda ix: F1.loc[ix.index.intersection(F1.index), "volT"].median())
cnt = raw_lbl.value_counts()
big = [c for c in cnt.index if cnt[c] >= MIN_CLUSTER]
for c in [c for c in cnt.index if cnt[c] < MIN_CLUSTER]:
    tgt = min(big, key=lambda b: abs(meds[b] - meds[c]) if np.isfinite(meds[c]) else -meds[b])
    print(f"folded a {cnt[c]}-member outlier cluster "
          f"({', '.join(raw_lbl[raw_lbl == c].index)}) into its volatility neighbour")
    raw_lbl[raw_lbl == c] = tgt
meds = raw_lbl.groupby(raw_lbl).apply(
    lambda ix: F1.loc[ix.index.intersection(F1.index), "volT"].median())
remap = {lab: i for i, lab in enumerate(meds.sort_values(ascending=False).index)}
LBL = raw_lbl.map(remap)
K_A = len(remap)

fig = make_subplots(rows=1, cols=K_A, shared_yaxes=True,
                    subplot_titles=[f"C{c} (n={(LBL == c).sum()})" for c in range(K_A)])
xs = np.arange(SER.shape[1])
for c in range(K_A):
    mem = np.where(LBL.to_numpy() == c)[0]
    for i in mem[:40]:
        fig.add_trace(go.Scatter(x=xs, y=SER[i], mode="lines", showlegend=False,
                                 line=dict(color=GRAY, width=0.5), opacity=0.35), row=1, col=c + 1)
    fig.add_trace(go.Scatter(x=xs, y=SER[mem].mean(axis=0), mode="lines", showlegend=False,
                             line=dict(color=C_CLUSTER[c], width=2.5)), row=1, col=c + 1)
fig.update_yaxes(title_text="daily vol (global z)", row=1, col=1)
style(fig, w=1250, h=330, title="Approach A: DTW k-means on daily volatility paths (fit month)")
fig.show()

medians = F1.loc[F1.index.intersection(LBL.index)].groupby(LBL).median().round(2)
medians["n"] = LBL.value_counts().sort_index()
print("cluster feature profiles (C0 = highest vol):")
display(medians)
for c in range(K_A):
    mem = LBL[LBL == c].index
    top = F1.loc[F1.index.intersection(mem), "log_qvol"].sort_values(ascending=False).index[:14]
    print(f"C{c} ({len(mem)}): " + ", ".join(t.replace('USDT', '') for t in top)
          + (" ..." if len(mem) > 14 else ""))

SER2, syms_a2 = vol_series(DAYS_H2)
km2 = TimeSeriesKMeans(n_clusters=K_A, metric="dtw", random_state=7, n_init=2, max_iter=30).fit(SER2)
common = [t for t in syms_a2 if t in LBL.index]
l2 = pd.Series(km2.labels_, index=syms_a2).loc[common]
print(f"split-half stability (refit on month 2 paths): ARI = "
      f"{adjusted_rand_score(LBL.loc[common], l2):.2f}")

### Approach B: Wasserstein clustering of the return distributions

In [ ]:
SK1, syms_b = [], []
for sym in CLUST_UNC:
    g = SUMM[sym]["h1"]
    if g is not None and "qsk" in g:
        SK1.append(g["qsk"]); syms_b.append(sym)
SK1 = np.array(SK1)
KM_B = KMeans(n_clusters=K_A, n_init=20, random_state=7).fit(SK1)
raw_b = pd.Series(KM_B.labels_, index=syms_b)
order_b = np.argsort(-raw_b.groupby(raw_b).apply(
    lambda ix: F1.loc[ix.index.intersection(F1.index), "volT"].median()).to_numpy())
LBL_B = raw_b.map({int(old): new for new, old in enumerate(order_b)})

fig = make_subplots(rows=1, cols=K_A, shared_yaxes=True,
                    subplot_titles=[f"B{c} (n={(LBL_B == c).sum()})" for c in range(K_A)])
for c in range(K_A):
    mem = np.where(LBL_B.to_numpy() == c)[0]
    for i in mem[:40]:
        fig.add_trace(go.Scatter(x=QSK_GRID, y=SK1[i], mode="lines", showlegend=False,
                                 line=dict(color=GRAY, width=0.5), opacity=0.35), row=1, col=c + 1)
    fig.add_trace(go.Scatter(x=QSK_GRID, y=SK1[mem].mean(axis=0), mode="lines", showlegend=False,
                             line=dict(color=C_CLUSTER[c], width=2.5)), row=1, col=c + 1)
    fig.update_xaxes(title_text="quantile", row=1, col=c + 1)
fig.update_yaxes(title_text="1m return (bps)", row=1, col=1)
style(fig, w=1250, h=330,
      title="Approach B: quantile functions of 1m returns by Wasserstein cluster (fit month)")
fig.show()

both = LBL.index.intersection(LBL_B.index)
print(f"A vs B agreement: ARI = {adjusted_rand_score(LBL.loc[both], LBL_B.loc[both]):.2f}")
print(pd.crosstab(LBL.loc[both].rename("A"), LBL_B.loc[both].rename("B")).to_string())
print("Approach A (DTW on volatility paths) is the operative partition below; B is the check.")

The full assignment table, one row per asset, both methods side by side. Shared colors mark shared cluster numbers (both methods order their clusters from highest to lowest volatility, so like colors mean like tiers).

In [ ]:
asgn = pd.DataFrame({"DTW": LBL.map(lambda c: f"C{c}"),
                     "Wasserstein": LBL_B.map(lambda c: f"C{c}")})
asgn = asgn.reindex(sorted(set(LBL.index) | set(LBL_B.index))).fillna("-")
cor_rows = pd.DataFrame({"DTW": "COR", "Wasserstein": "COR"}, index=COR)
asgn = pd.concat([cor_rows, asgn.drop(index=[t for t in COR if t in asgn.index])])
asgn.index = [t.replace("USDT", "") for t in asgn.index]
asgn = pd.concat([asgn[asgn.DTW == "COR"], asgn[asgn.DTW != "COR"].sort_values(["DTW", "Wasserstein"])])
CLR = {f"C{i}": c + "55" for i, c in enumerate(C_CLUSTER)}
CLR["COR"] = C_COR + "55"
sty = asgn.style.map(lambda v: f"background-color: {CLR.get(v, '')}")
with pd.option_context("styler.render.max_elements", 10**6):
    display(sty)

## 5. Exact direct[1h] caps for a representative subset

Caps are computed with the estimator from the penalty cap notebooks: non overlapping 26 second window moves on a 1 second last price grid, cap at time t equal to the target quantile of the trailing 138 windows, applied strictly to the next window. The price grid comes from 1 second klines, which are the same object the tick pipeline produces after resampling, minus the outlier cleaning; the difference is measured directly in the validation below.

The subset spans each cluster rather than sampling only its liquid members: per cluster the members at the volatility extremes and median, the two largest by volume, and two random draws, plus the pairs already used in the penalty notebooks and the assets that dominate CoW flow. Every subset member contributes its cap series to its cluster's percentile. Pairs whose tick grid is coarse relative to their own tail (tick above half the 2 percent quantile, or mostly moveless bars) additionally get a tick floor when receiving: the borrowed cap is never below 1.5 times their tick, since a cap below one price increment cannot be enforced.

In [ ]:
FORCED = [s for s in ("ETHUSDT SOLUSDT DOGEUSDT BNBUSDT POLUSDT AVAXUSDT "
                      "BTCUSDT LINKUSDT UNIUSDT AAVEUSDT PEPEUSDT ONDOUSDT CRVUSDT "
                      "COWUSDT BABYUSDT KAITOUSDT WBETHUSDT").split()
          if s in LBL.index]
FORCED += [t for t in COR if t.endswith("USDT")]           # every real correlated pair
# pinned when the tick grid is coarse relative to the pair's own tail: quantization then
# dominates the low quantile, regardless of the absolute tick size
PINNED = set(F1.index[(F1.tick_bps > 0.5*F1.q_tail) | (F1.zero_share > 0.7)])
picks = set(FORCED)
for c in range(K_A):
    mem = F1.loc[LBL[LBL == c].index.intersection(F1.index)]
    byv = mem.volT.sort_values()
    picks |= {byv.index[int(round(f*(len(byv) - 1)))] for f in np.linspace(0, 1, 48)}
    picks |= set(mem.log_qvol.sort_values(ascending=False).index[:12])
    pool = [s for s in mem.index if s not in picks]
    if pool: picks |= set(rng.choice(pool, size=min(30, len(pool)), replace=False))
SUBSET = sorted(t for t in picks if " " not in t)           # synthetics get caps from their legs
if len(SUBSET) > N_BUDGET:
    drop = [s for s in SUBSET if s not in FORCED]
    drop = list(F1.loc[drop, "log_qvol"].sort_values().index)
    SUBSET = sorted(set(SUBSET) - set(drop[:len(SUBSET) - N_BUDGET]))
SYN_LEGS = ["ETHUSDT", "WBETHUSDT", "COWUSDT", "BABYUSDT", "KAITOUSDT"]
EXTRA_1S = sorted(set(["USDCUSDT", "HYPEUSDT", "WBETHETH", "DOGEUSDT", "POLUSDT"] + SYN_LEGS) - set(SUBSET))
print(f"subset: {len(SUBSET)} symbols ({sum(s in PINNED for s in SUBSET)} tick-floored) "
      f"+ extras {EXTRA_1S}")
for c in range(K_A):
    ss = [s.replace("USDT", "") for s in SUBSET if LBL.get(s) == c]
    print(f"  C{c} ({len(ss)}): " + ", ".join(ss))
print(f"  COR ({len(COR)}): " + ", ".join(t.replace("USDT", "") for t in COR))
est_mb = (len(SUBSET) + len(EXTRA_1S))*len(DAYS)*0.9
print(f"download estimate: ~{est_mb/1000:.1f} GB of 1s klines")

In [ ]:
def files_1s(sym):
    return [(f"https://data.binance.vision/data/spot/daily/klines/{sym}/1s/{sym}-1s-{d}.zip",
             os.path.join(CACHE_1S, f"{sym}-1s-{d}.zip")) for d in DAYS]
jobs = [uj for sym in SUBSET + EXTRA_1S for uj in files_1s(sym)]
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as ex:
    for i, _ in enumerate(ex.map(lambda a: dl(*a), jobs)):
        if i % 500 == 0: print(f"  {i}/{len(jobs)} files, {time.time() - t0:.0f}s", flush=True)
print(f"1s downloads done in {time.time() - t0:.0f}s")

In [ ]:
N_WIN = (86400 - 1)//T_EXCL                      # windows per day
WIN_TS = pd.DatetimeIndex([pd.Timestamp(d) + pd.Timedelta(seconds=(i + 1)*T_EXCL)
                           for d in DAYS for i in range(N_WIN)])   # tz-naive UTC; kaleido rejects tz-aware
H2_MASK = np.repeat(np.isin(DAYS, DAYS_H2), N_WIN)

def day_px_1s(sym, d):
    path = os.path.join(CACHE_1S, f"{sym}-1s-{d}.zip")
    if not os.path.exists(path): return None
    try:
        with zipfile.ZipFile(path) as z:
            name = z.namelist()[0]
            hh = not z.open(name).read(16)[:1].isdigit()
            df = pd.read_csv(z.open(name), header=0 if hh else None,
                             usecols=[0, 4], names=["ts", "close"])
    except Exception:
        return None
    unit = "us" if df["ts"].iloc[0] > 10**14 else "ms"
    sec = ((df["ts"] // (10**6 if unit == "us" else 10**3)) % 86400).to_numpy()
    px = np.full(86400, np.nan)
    px[sec] = df["close"].to_numpy()
    return pd.Series(px).ffill().bfill().to_numpy()

def px_moves(px):
    p0 = px[: N_WIN*T_EXCL: T_EXCL]
    p1 = px[T_EXCL: N_WIN*T_EXCL + 1: T_EXCL]
    return ((p1/p0 - 1.0)*1e4).astype(np.float32)

# synthetic pairs: divide the legs' 1s price grids, then run the identical estimator
SPECIALS = {"WBETHETH": ("WBETHETH", None),
            "WBETH/ETH syn": (None, ("WBETHUSDT", "ETHUSDT")),
            "COW/ETH syn": (None, ("COWUSDT", "ETHUSDT")),
            "BABY/KAITO syn": (None, ("BABYUSDT", "KAITOUSDT"))}

def sym_arrays(name, sym=None, legs=None):
    slug = "".join(ch for ch in name if ch.isalnum())
    f_mv = os.path.join(CACHE_NPY, f"{slug}_mv_T{T_EXCL}.npy")
    f_cp = os.path.join(CACHE_NPY, f"{slug}_cap_T{T_EXCL}_q{int(Q_TARGET*1000)}.npy")
    if os.path.exists(f_mv) and os.path.exists(f_cp):
        return np.load(f_mv), np.load(f_cp)
    days_mv = []
    for d in DAYS:
        if legs is None:
            px = day_px_1s(sym, d)
        else:
            pa, pb = day_px_1s(legs[0], d), day_px_1s(legs[1], d)
            px = pa/pb if pa is not None and pb is not None else None
        days_mv.append(px_moves(px) if px is not None else np.full(N_WIN, np.nan, np.float32))
    mv = np.concatenate(days_mv)
    cap = (-pd.Series(mv).rolling(W_1H, min_periods=W_1H).quantile(Q_TARGET)
           .shift(1).to_numpy().astype(np.float32))
    np.save(f_mv, mv); np.save(f_cp, cap)
    return mv, cap

MOVES, CAPS = {}, {}
t0 = time.time()
todo = [(t, t, None) for t in SUBSET + EXTRA_1S] +        [(n, sm, lg) for n, (sm, lg) in SPECIALS.items() if n not in SUBSET + EXTRA_1S]
for i, (name, sym, legs) in enumerate(todo):
    MOVES[name], CAPS[name] = sym_arrays(name, sym=sym, legs=legs)
    if i % 8 == 0: print(f"  {i}/{len(todo)} series, {time.time() - t0:.0f}s", flush=True)
MOVES = pd.DataFrame(MOVES, index=WIN_TS)
CAPS = pd.DataFrame(CAPS, index=WIN_TS)
print(f"caps built in {time.time() - t0:.0f}s | grid {len(WIN_TS):,} windows x {CAPS.shape[1]} symbols")
own = {s: float(np.mean(MOVES[s].to_numpy()[H2_MASK] < -CAPS[s].to_numpy()[H2_MASK]))
       for s in SUBSET if s in CAPS}
print(f"own-cap revert rate, eval month: median {np.median(list(own.values())):.2%} "
      f"(target {Q_TARGET:.0%}; the small overshoot matches the penalty notebooks)")

### Correlated and synthetic pairs

The agreed rule for a pair without its own listing is to synthesize its price from the legs and run the identical estimator. Three cases at the 8 percent target: the listed correlated pair WBETHETH, the synthetic COW/ETH, and the synthetic BABY/KAITO. WBETH/ETH exists both as a real listing and as a synthetic from WBETHUSDT and ETHUSDT, so it measures how much the synthetic construction distorts the cap.

In [ ]:
rows = []
for name in list(SPECIALS) + ["ETHUSDT", "USDCUSDT"]:
    if name not in CAPS.columns: continue
    mv, cp = MOVES[name].to_numpy()[H2_MASK], CAPS[name].to_numpy()[H2_MASK]
    ok = np.isfinite(mv) & np.isfinite(cp)
    rows.append(dict(pair=name, mean_cap_bps=float(np.nanmean(cp)),
                     rev_rate=float(np.mean(mv[ok] < -cp[ok])),
                     zero_move_share=float(np.mean(mv[ok] == 0.0))))
sp = pd.DataFrame(rows).set_index("pair")
print(sp.round(3).to_string())
if "WBETHETH" in CAPS.columns and "WBETH/ETH syn" in CAPS.columns:
    d = np.abs(CAPS["WBETH/ETH syn"] - CAPS["WBETHETH"])[H2_MASK]
    print(f"\nsynthetic vs real WBETH/ETH: median |cap difference| = {np.nanmedian(d):.2f} bps, "
          f"95th pct = {np.nanpercentile(d.dropna(), 95):.2f} bps")

### Why the synthetic runs hotter than the real pair

The construction is the agreed one: divide the legs' price series on a common grid, then run the identical estimator on the resulting rate. The inflation comes from the data, not the construction. Each leg's last trade price updates at its own moments and carries its own bid to ask bounce; when the legs do not print together, stale prices and two independent noise terms enter the ratio and do not cancel. The real WBETHETH order book quotes the exchange rate directly, so its price only moves when the rate itself trades, and it stays pinned to its tick grid for long stretches. The consequence for caps: a synthetic cap is an upper bound, conservative and acceptable for pairs with no listing, but wherever a real listing exists it dominates, which is why the synthetic never contributes to a group cap when its real counterpart is a member.

The figure is interactive; zooming either panel zooms both.

In [ ]:
last30 = pd.Timestamp(DAYS[-30])
CH = {k: CLOSE_1M[k][CLOSE_1M[k].index >= last30].iloc[::3]
      for k in ("WBETHETH", "WBETH/ETH syn", "WBETHUSDT", "ETHUSDT")}
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=["WBETH/ETH: real listing vs synthetic ratio",
                                    "the legs: WBETHUSDT and ETHUSDT"])
fig.add_trace(go.Scatter(x=CH["WBETHETH"].index, y=CH["WBETHETH"].to_numpy(),
                         name="WBETHETH (real)", line=dict(color=C_COR, width=1.4)), row=1, col=1)
fig.add_trace(go.Scatter(x=CH["WBETH/ETH syn"].index, y=CH["WBETH/ETH syn"].to_numpy(),
                         name="synthetic (WBETHUSDT / ETHUSDT)",
                         line=dict(color=CAP_ACC, width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=CH["WBETHUSDT"].index, y=CH["WBETHUSDT"].to_numpy(),
                         name="WBETHUSDT", line=dict(color="#2a78d6", width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=CH["ETHUSDT"].index, y=CH["ETHUSDT"].to_numpy(),
                         name="ETHUSDT", line=dict(color=INK, width=1)), row=2, col=1)
fig.update_yaxes(title_text="WBETH/ETH rate", row=1, col=1)
fig.update_yaxes(title_text="price (USDT)", row=2, col=1)
style(fig, w=1150, h=650, title="Real vs synthetic WBETH/ETH, last 30 days (3 minute sampling)")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0))
fig.show(renderer="plotly_mimetype+notebook")
print(f"points per trace: {len(CH['WBETHETH'])} real / {len(CH['WBETH/ETH syn'])} synthetic")

### Are 1 second klines an admissible stand in for ticks

Acceptance criteria set before looking: median absolute cap difference below 0.3 bps and revert rate difference below 0.2 percentage points on the pairs where the tick pipeline is available. USDCUSDT is included deliberately as the degenerate pegged case.

In [ ]:
VAL_SYMS = [s for s in ("ETHUSDT", "DOGEUSDT", "POLUSDT", "USDCUSDT") if s in CAPS.columns]
VAL_DAYS = [d for d in DAYS_H2 if d <= "2026-07-16"]      # tick cache coverage

def clean_ticks(df, k=25, mad_mult=3.0, floor_ticks=2.0, floor_bps=1.0):
    df = df[(df.price > 0) & df.price.notna()].drop_duplicates(subset=["t", "price"]).copy()
    tick = np.median(np.diff(np.unique(np.sort(df.price.values))))
    med = df.price.rolling(2*k + 1, center=True, min_periods=k).median()
    mad = (df.price - med).abs().rolling(2*k + 1, center=True, min_periods=k).median()
    floor_abs = max(floor_ticks*tick, df.price.median()*floor_bps*1e-4)
    keep = ((df.price - med).abs() <= mad_mult*mad + floor_abs).fillna(True)
    return df[keep].set_index("t")

def day_moves_tick(sym, d):
    f_np = os.path.join(CACHE_NPY, f"{sym}_tickmv_{d}_T{T_EXCL}.npy")
    if os.path.exists(f_np): return np.load(f_np)
    path = os.path.join(CACHE_TICKS, f"{sym}-aggTrades-{d}.zip")
    if not os.path.exists(path): return None
    with zipfile.ZipFile(path) as z:
        name = z.namelist()[0]
        hh = not z.open(name).read(64)[:1].isdigit()
        cols = ["aggId", "price", "qty", "firstId", "lastId", "ts", "isBuyerMaker", "isBestMatch"]
        df = pd.read_csv(z.open(name), header=0 if hh else None, names=None if hh else cols)
        if hh:
            df.columns = [str(c).lower() for c in df.columns]
            tcol = [c for c in df.columns if "time" in c or c == "ts" or "transact" in c][0]
            df = df.rename(columns={tcol: "ts"})
    df = df[["price", "ts"]].astype({"ts": "int64", "price": float})
    unit = "us" if df["ts"].iloc[0] > 10**14 else "ms"
    df["t"] = pd.to_datetime(df["ts"], unit=unit, utc=True)
    px = clean_ticks(df[["t", "price"]]).price.resample("1s").last().ffill()
    px = px.reindex(pd.date_range(d, periods=86400, freq="1s", tz="UTC")).ffill().bfill().to_numpy()
    p0 = px[: N_WIN*T_EXCL: T_EXCL]
    p1 = px[T_EXCL: N_WIN*T_EXCL + 1: T_EXCL]
    mv = ((p1/p0 - 1.0)*1e4).astype(np.float32)
    np.save(f_np, mv)
    return mv

rows, t0 = [], time.time()
for sym in VAL_SYMS:
    seed = day_moves_tick(sym, DAYS_H1[-1])
    mvs = [seed] + [day_moves_tick(sym, d) for d in VAL_DAYS]
    if any(m is None for m in mvs): continue
    mt = np.concatenate(mvs)
    ct = (-pd.Series(mt).rolling(W_1H, min_periods=W_1H).quantile(Q_TARGET).shift(1)
          .to_numpy()[N_WIN:])
    mt = mt[N_WIN:]
    sel = np.isin(DAYS, VAL_DAYS)
    mk = np.repeat(sel, N_WIN)
    ck, mk1 = CAPS[sym].to_numpy()[mk], MOVES[sym].to_numpy()[mk]
    d = np.abs(ct - ck)
    rows.append(dict(pair=sym, med_dcap=np.nanmedian(d), p95_dcap=np.nanpercentile(d, 95),
                     rev_tick=float(np.mean(mt < -ct)), rev_1s=float(np.mean(mk1 < -ck))))
    print(f"  {sym} done, {time.time() - t0:.0f}s", flush=True)
vt = pd.DataFrame(rows).set_index("pair")
vt["d_rev_pp"] = (vt.rev_1s - vt.rev_tick)*100
display(vt.round(3))
okv = vt.drop(index="USDCUSDT", errors="ignore")
verdict = (okv.med_dcap < 0.3).all() and (okv.d_rev_pp.abs() < 0.2).all()
print("verdict:", "1s klines admissible for non-pegged pairs" if verdict
      else "criteria NOT met, results below should be read as approximate")

## 6. Cluster cap distributions over time

The figure below is the core exhibit. Per cluster: the band of member caps (10th to 90th and 25th to 75th percentile across members at each point in time), the member median, and in bold the 75th percentile cluster cap that the proposal would apply to assets without their own feed. Tight bands mean a cluster cap mis-sizes its members by little; wide bands mean the cluster is not a usable cap group. Reading rule fixed in advance: a cluster whose members' mean caps differ by more than a factor 3 between the 90th and 10th percentile member fails, and per asset caps or a finer split are needed there.

In [ ]:
CONTRIB = {c: [s for s in CAPS.columns if LBL.get(s) == c] for c in range(K_A)}
CONTRIB["COR"] = [s for s in CAPS.columns if s in COR and s != "WBETH/ETH syn"]  # synthetic receive-only
KEYS = list(range(K_A)) + ["COR"]
GRP = MOVES.index.floor("15min")
CAPS15 = CAPS.groupby(GRP).mean()
titles = [f"C{c}: {len(CONTRIB[c])} contributing of {int((LBL == c).sum())} members" for c in range(K_A)]
titles.append(f"COR (rule based): {len(CONTRIB['COR'])} contributing of {len(COR)} members")
fig = make_subplots(rows=len(KEYS), cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    subplot_titles=titles)
for r0, c in enumerate(KEYS):
    mem = CAPS15[CONTRIB[c]]
    if mem.shape[1] < 2: continue
    if c == "COR":
        for t in CONTRIB["COR"]:
            fig.add_trace(go.Scatter(x=mem.index, y=mem[t], mode="lines", showlegend=False,
                                     line=dict(color=C_COR, width=1), opacity=0.6,
                                     hoverinfo="skip"), row=r0 + 1, col=1)
        fig.add_trace(go.Scatter(x=mem.index, y=mem.quantile(CLUSTER_PCTL/100, axis=1),
                                 mode="lines", showlegend=False,
                                 line=dict(color=CAP_ACC, width=2.2)), row=r0 + 1, col=1)
    else:
        q = mem.quantile([0.10, 0.25, 0.50, 0.75, 0.90], axis=1).T
        x = q.index
        for lo, hi, op in ((0.10, 0.90, 0.16), (0.25, 0.75, 0.30)):
            fig.add_trace(go.Scatter(x=x, y=q[hi], mode="lines", line=dict(width=0),
                                     showlegend=False, hoverinfo="skip"), row=r0 + 1, col=1)
            fig.add_trace(go.Scatter(x=x, y=q[lo], mode="lines", line=dict(width=0), fill="tonexty",
                                     fillcolor=f"rgba{tuple(int(C_CLUSTER[c][i:i+2], 16) for i in (1, 3, 5)) + (op,)}",
                                     showlegend=False, hoverinfo="skip"), row=r0 + 1, col=1)
        fig.add_trace(go.Scatter(x=x, y=q[0.50], mode="lines", showlegend=(r0 == 0), name="member median",
                                 line=dict(color=GRAY, width=1)), row=r0 + 1, col=1)
        fig.add_trace(go.Scatter(x=x, y=q[CLUSTER_PCTL/100], mode="lines", showlegend=(r0 == 0),
                                 name=f"{CLUSTER_PCTL}th pct cluster cap",
                                 line=dict(color=CAP_ACC, width=2.2)), row=r0 + 1, col=1)
    fig.update_yaxes(title_text="cap (bps)", row=r0 + 1, col=1)
fig.add_vline(x=pd.Timestamp(DAYS_H2[0]).to_pydatetime(), line=dict(color=GRAY, dash="dot", width=1))
style(fig, w=1150, h=230*len(KEYS) + 140,
      title=f"Member cap distribution and the {CLUSTER_PCTL}th percentile group cap "
            f"(COR drawn as individual members; dotted line: eval month starts)")
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.008, x=0))
fig.show()

meancap = CAPS.loc[:, [t for c in range(K_A) for t in CONTRIB[c]]][H2_MASK].mean()
disp = pd.DataFrame({c: dict(
    p90_p10=meancap[CONTRIB[c]].quantile(0.9)/max(meancap[CONTRIB[c]].quantile(0.1), 1e-9),
    med_bps=meancap[CONTRIB[c]].median(),
    p75_bps=meancap[CONTRIB[c]].quantile(0.75)) for c in range(K_A) if len(CONTRIB[c]) >= 2}).T
disp["verdict"] = np.where(disp.p90_p10 <= 3, "usable", "TOO WIDE")
print("within-cluster dispersion of member mean caps (eval month, statistical clusters):")
display(disp.round(2))
corm = CAPS[CONTRIB["COR"]].to_numpy()[H2_MASK]
corcaps = np.nanmean(corm, axis=0)
print(f"COR: {len(CONTRIB['COR'])} contributors, member mean caps "
      f"{np.nanmin(corcaps):.2f} - {np.nanmax(corcaps):.2f} bps, 75th pct "
      f"{np.nanpercentile(corcaps, 75):.2f} bps (small group, rule based, no dispersion verdict)")

## 6b. Benchmark: one constant cap per CoW fee class

The fixed cap option in the CIP needs a benchmark: one constant per class, using CoW's own fee grouping (correlated when both tokens share a CMS bucket, standard otherwise). The standard class constant is the empirical May calibrated constant of the penalty notebooks, recomputed here for this notebook's window length from the tick pipeline: the pooled 8 percent quantile of window moves over May for the six liquid pairs. The correlated class has no tick analog in that methodology, so its constant stays calibrated on the fit month of this panel. As before, the stable bucket deliberately mixes USD stables, EUR stables and tokenized equities, so a EUR to USD pair counts as correlated while carrying real FX volatility; the per pair spread below shows what that costs.

In [ ]:
SIX_LIQ = ["ETHUSDT", "SOLUSDT", "DOGEUSDT", "BNBUSDT", "POLUSDT", "AVAXUSDT"]
DAYS_MAY = [f"2026-05-{d:02d}" for d in range(1, 31)]
def fetch_tick(sym, d):
    fn = f"{sym}-aggTrades-{d}.zip"
    path = os.path.join(CACHE_TICKS, fn)
    if not os.path.exists(path):
        try:
            urllib.request.urlretrieve(
                f"https://data.binance.vision/data/spot/daily/aggTrades/{sym}/{fn}", path)
        except Exception:
            pass
    return path
pool = []
t0 = time.time()
for sym in SIX_LIQ:
    for d in DAYS_MAY:
        fetch_tick(sym, d)
        mv = day_moves_tick(sym, d)
        if mv is not None: pool.append(mv)
    print(f"  {sym} May tick moves done, {time.time() - t0:.0f}s", flush=True)
CONST_POOL = -float(np.quantile(np.concatenate(pool), Q_TARGET))
print(f"const[May, T={T_EXCL}s] = {CONST_POOL:.2f} bps | pooled {Q_TARGET:.0%} quantile of "
      f"non-overlapping {T_EXCL}s moves, MAD-cleaned aggTrades on a 1s grid, 2026-05-01..30, "
      "six liquid pairs, per the CONST_MAY methodology of the penalty notebooks")

CF_CLASS = {n: cow_class(n) for n in CAPS.columns}
H1_MASK = ~H2_MASK
CF_CAP, CF_REV = {}, {}
corr_mem = [n for n, c in CF_CLASS.items() if c == "correlated"]
pc = MOVES[corr_mem].to_numpy()[H1_MASK].ravel()
CONST_CORR = -float(np.quantile(pc[np.isfinite(pc)], Q_TARGET))
std_mem = [n for n, c in CF_CLASS.items() if c == "standard"]
ps = MOVES[std_mem].to_numpy()[H1_MASK].ravel()
CONST_STD_FIT = -float(np.quantile(ps[np.isfinite(ps)], Q_TARGET))
for cls, const, mem in (("correlated", CONST_CORR, corr_mem), ("standard", CONST_POOL, std_mem)):
    rates = []
    for n in mem:
        mv = MOVES[n].to_numpy()[H2_MASK]
        mv = mv[np.isfinite(mv)]
        if not mv.size: continue
        CF_CAP[n] = const
        CF_REV[n] = float(np.mean(mv < -const))
        rates.append(CF_REV[n])
    print(f"{cls:11s} n = {len(rates):3d} | constant = {const:6.2f} bps | eval-month revert: "
          f"median {np.median(rates):.2%}, 10-90th pct {np.percentile(rates, 10):.2%} - "
          f"{np.percentile(rates, 90):.2%}, max {max(rates):.2%}")
print(f"for continuity: the fit-month pooled standard constant of this panel would be "
      f"{CONST_STD_FIT:.2f} bps")

fig = go.Figure()
for cls, col in (("standard", "#2a78d6"), ("correlated", C_COR)):
    r = [CF_REV[n] for n, c in CF_CLASS.items() if c == cls and n in CF_REV]
    if r: fig.add_trace(go.Histogram(x=r, xbins=dict(size=0.01), name=f"{cls} (n={len(r)})",
                                     marker_color=col, opacity=0.65))
fig.add_vline(x=Q_TARGET, line=dict(color=INK, dash="dash", width=1.2))
fig.update_xaxes(title="eval-month revert rate under the class constant", tickformat=".0%")
fig.update_yaxes(title="pairs")
style(fig, w=850, h=420, title="One constant cap per CoW fee class: realized revert rates "
                               f"(dashed: {Q_TARGET:.0%} target)")
fig.update_layout(barmode="overlay")
fig.show()

In [ ]:
TB_ALL = {**F1.tick_bps.to_dict(), **TICK_BPS}
rows = []
for key in ["COR"] + list(range(K_A)):
    lab = "COR" if key == "COR" else f"C{key}"
    mem_all = [t for t in CAPS.columns if (t in COR if key == "COR" else LBL.get(t) == key)]
    contrib = CONTRIB[key]
    if len(contrib) < 3: continue
    M = CAPS[contrib].to_numpy()
    for t in mem_all:
        cols = [j for j, u in enumerate(contrib) if u != t]
        if len(cols) < 2: continue
        loo75 = np.nanpercentile(M[:, cols], 75, axis=1)
        loo50 = np.nanpercentile(M[:, cols], 50, axis=1)
        floor = 1.5*TB_ALL.get(t, 0.0) if t in PINNED or (key == "COR" and TB_ALL.get(t, 0) > 0.5) else 0.0
        l75, l50 = np.maximum(loo75, floor), np.maximum(loo50, floor)
        mv, cp = MOVES[t].to_numpy()[H2_MASK], CAPS[t].to_numpy()[H2_MASK]
        mvf = mv[np.isfinite(mv)]
        if not mvf.size: continue
        r_own = float(np.mean(mv < -cp))
        rows.append(dict(pair=t, cluster=lab, pinned=t in PINNED, tick_bps=TB_ALL.get(t, np.nan),
                         own_cap=np.nanmean(cp), loo75_raw=np.nanmean(loo75[H2_MASK]),
                         floor_bps=floor, loo75_cap=np.nanmean(l75[H2_MASK]),
                         loo50_cap=np.nanmean(l50[H2_MASK]), rev_own=r_own,
                         rev_loo75=float(np.mean(mv < -l75[H2_MASK])),
                         rev_loo50=float(np.mean(mv < -l50[H2_MASK]))))
val = pd.DataFrame(rows).set_index("pair")
val["cap_ratio"] = val.loo75_cap/val.own_cap
val["rev_ratio"] = val.rev_loo75/val.rev_own.replace(0, np.nan)
val = val.sort_values(["cluster", "rev_loo75"])
extra_rows = []
for name in CAPS.columns:
    if name in val.index: continue
    mv, cp = MOVES[name].to_numpy()[H2_MASK], CAPS[name].to_numpy()[H2_MASK]
    ok = np.isfinite(mv) & np.isfinite(cp)
    if not ok.any(): continue
    kind = ("COR" if name in COR else "synthetic" if "syn" in name
            else (f"C{LBL.get(name)}" if LBL.get(name) is not None
                  else ("young" if name in RECENT else "unclustered")))
    extra_rows.append(dict(pair=name, cluster=kind, pinned=name in PINNED,
                           tick_bps=TB_ALL.get(name, np.nan), own_cap=float(np.nanmean(cp)),
                           rev_own=float(np.mean(mv[ok] < -cp[ok]))))
val_csv = pd.concat([val, pd.DataFrame(extra_rows).set_index("pair")]) if extra_rows else val
val_csv["cow_class"] = [CF_CLASS.get(p, "") for p in val_csv.index]
val_csv["cf_const_cap"] = [CF_CAP.get(p, np.nan) for p in val_csv.index]
val_csv["cf_rev"] = [CF_REV.get(p, np.nan) for p in val_csv.index]
val_csv.round(4).to_csv(f"clustering_loo_caps_8pct_T{T_EXCL}.csv")
print(f"full table saved to clustering_loo_caps_8pct_T{T_EXCL}.csv "
      f"({len(val)} grouped pairs + {len(extra_rows)} extras)")
with pd.option_context("display.max_rows", None):
    print(val.round(3).to_string())
summ = val.groupby("cluster").agg(n=("rev_loo75", "size"),
                                  share_le_target=("rev_loo75", lambda x: float(np.mean(x <= Q_TARGET))),
                                  worst_rev=("rev_loo75", "max"), med_cap_ratio=("cap_ratio", "median"),
                                  med_rev_ratio=("rev_ratio", "median"))
print("per-cluster summary (eval month, 75th percentile group cap):")
print(summ.round(3).to_string())

fig = go.Figure()
for key in ["COR"] + list(range(K_A)):
    lab = "COR" if key == "COR" else f"C{key}"
    m = val[val.cluster == lab]
    if not len(m): continue
    col = C_COR if key == "COR" else C_CLUSTER[key]
    fig.add_trace(go.Scatter(x=m.cap_ratio, y=m.rev_loo75, mode="markers+text",
                             text=[p.replace("USDT", "") for p in m.index], textposition="top center",
                             textfont=dict(size=8), name=lab,
                             marker=dict(size=8, color=col)))
fig.add_hline(y=Q_TARGET, line=dict(color=INK, dash="dash", width=1.2))
fig.add_vline(x=1.0, line=dict(color=GRAY, dash="dot", width=1))
fig.update_xaxes(title="group cap / own cap (mean)")
fig.update_yaxes(title="revert rate under LOO group cap", tickformat=".1%")
style(fig, w=1000, h=540, title=f"Cost of borrowing the 75th percentile group cap, eval month "
                                f"(dashed: {Q_TARGET:.0%} target)")
fig.show()

In [ ]:
rows = []
for pct in PCTL_GRID:
    rates, capbps = [], []
    for c in range(K_A):
        contrib = CONTRIB[c]
        if len(contrib) < 3: continue
        M = CAPS[contrib].to_numpy()
        for s in [t for t in SUBSET if LBL.get(t) == c and t in CAPS.columns]:
            cols = [j for j, t in enumerate(contrib) if t != s]
            if len(cols) < 2: continue
            loo = np.nanpercentile(M[:, cols], pct, axis=1)[H2_MASK]
            rates.append(float(np.mean(MOVES[s].to_numpy()[H2_MASK] < -loo)))
            capbps.append(float(np.nanmean(loo)))
    rows.append(dict(pctl=pct, share_le_target=float(np.mean(np.array(rates) <= Q_TARGET)),
                     worst_rev=max(rates), mean_cap_bps=float(np.mean(capbps))))
fr = pd.DataFrame(rows).set_index("pctl")
display(fr.round(3))
print("COR is rule based and outside the percentile dial.")
print("the percentile is the one governance dial: higher protects more members at the price "
      "of a looser (weaker) cap for everyone in the cluster.")

## 8. The default for unknown tokens

A token nobody has data for starts on the highest cap cluster's 75th percentile cap. Applying that default to every subset member shows what it costs: essentially no reverts for anything calmer than the top cluster, meaning the penalty loses its bite until the token graduates. Proposed graduation: after 7 days of any usable feed the token's features place it in a cluster; once it has its own feed it gets its own direct[1h] cap. The 7 is a parameter, not a finding.

In [ ]:
cmax = int(disp.p75_bps.idxmax()) if len(disp) else 0
dflt = np.nanpercentile(CAPS[CONTRIB[cmax]].to_numpy(), CLUSTER_PCTL, axis=1)
print(f"default = C{cmax} {CLUSTER_PCTL}th pct cap | eval-month mean {np.nanmean(dflt[H2_MASK]):.1f} bps, "
      f"range {np.nanmin(dflt[H2_MASK]):.1f} - {np.nanmax(dflt[H2_MASK]):.1f} bps")
rows = []
for s in SUBSET:
    if s not in CAPS.columns: continue
    mv = MOVES[s].to_numpy()[H2_MASK]
    rows.append(dict(pair=s, cluster=f"C{LBL.get(s)}",
                     rev_default=float(np.mean(mv < -dflt[H2_MASK]))))
dv = pd.DataFrame(rows).groupby("cluster").rev_default.agg(["median", "max"])
print("revert rates under the default cap, eval month:")
display(dv.round(4))
if "HYPEUSDT" in MOVES.columns:
    mv = MOVES["HYPEUSDT"].to_numpy(); ok = np.isfinite(mv) & np.isfinite(dflt)
    if ok.any():
        print(f"young-asset example HYPEUSDT: revert rate under default over its live window = "
              f"{float(np.mean(mv[ok] < -dflt[ok])):.2%} "
              f"(own-cap rate {float(np.mean(mv[ok] < -CAPS['HYPEUSDT'].to_numpy()[ok])):.2%})")
    else:
        print("young-asset example HYPEUSDT: no 1s data available in the window, example skipped")

## Appendix: how far 1 minute klines can stretch

Full universe coverage without 1 second data would need caps approximated from 1 minute klines, scaled from 60 to 26 seconds by the square root of time. Comparing that approximation with the exact caps on the subset shows where it holds and where it breaks; the penalty notebooks predict it breaks exactly on tick pinned pairs, whose tails do not scale.

In [ ]:
def cap1m_mean(sym):
    frames = []
    for _, path in files_1m(sym):
        if not os.path.exists(path): continue
        try:
            with zipfile.ZipFile(path) as z:
                name = z.namelist()[0]
                hh = not z.open(name).read(16)[:1].isdigit()
                frames.append(pd.read_csv(z.open(name), header=0 if hh else None,
                                          usecols=[0, 4], names=["ts", "close"]))
        except Exception: continue
    if not frames: return np.nan
    df = pd.concat(frames).sort_values("ts")
    unit = "us" if df["ts"].iloc[0] > 10**14 else "ms"
    day = pd.to_datetime(df["ts"], unit=unit, utc=True).dt.strftime("%Y-%m-%d")
    df = df[(day >= DATE_START) & (day <= DATE_END)]
    r = (df.close.pct_change()*1e4).to_numpy()[1:]
    cap = (-pd.Series(r).rolling(1440, min_periods=1440).quantile(Q_TARGET).shift(1)
           .to_numpy())*np.sqrt(T_EXCL/60)
    n2 = int(len(r)*(len(DAYS_H1)/len(DAYS)))
    return float(np.nanmean(cap[n2:]))

rows = []
for s in SUBSET:
    if s not in CAPS.columns: continue
    ex = float(np.nanmean(CAPS[s].to_numpy()[H2_MASK]))
    ap = cap1m_mean(s)
    rows.append(dict(pair=s, exact=ex, approx=ap, ratio=ap/ex if ex else np.nan,
                     pinned=s in PINNED))
ap = pd.DataFrame(rows).set_index("pair").dropna()
fig = go.Figure()
for pin, col, nm in ((False, "#2a78d6", "liquid"), (True, "#c22f2f", "tick pinned")):
    m = ap[ap.pinned == pin]
    fig.add_trace(go.Scatter(x=m.exact, y=m.approx, mode="markers+text",
                             text=[p.replace("USDT", "") for p in m.index],
                             textposition="top center", textfont=dict(size=8),
                             name=nm, marker=dict(size=8, color=col)))
lim = [ap[["exact", "approx"]].min().min()*0.8, ap[["exact", "approx"]].max().max()*1.2]
fig.add_trace(go.Scatter(x=lim, y=lim, mode="lines", showlegend=False,
                         line=dict(color=GRAY, dash="dot", width=1)))
fig.update_xaxes(title="exact direct[1h] cap, mean bps")
fig.update_yaxes(title="1m approximation, mean bps")
style(fig, w=820, h=520, title="1m sqrt-scaled approximation vs exact cap (eval month)")
fig.show()
print(f"approximation ratio: liquid median {ap[~ap.pinned].ratio.median():.2f}, "
      f"tick-pinned median {ap[ap.pinned].ratio.median() if ap.pinned.any() else float('nan'):.2f}")

In [ ]:
print(f"headline block (identical format in both notebooks)")
print(f"  T_EXCL {T_EXCL}s | windows/hour {W_1H} | windows/day {N_WIN}")
if "ETHUSDT" in CAPS.columns:
    print(f"  ETHUSDT own direct[1h] cap, eval mean: "
          f"{float(np.nanmean(CAPS['ETHUSDT'].to_numpy()[H2_MASK])):.2f} bps")
for c in range(K_A):
    if c in disp.index:
        print(f"  C{c}: member median {disp.loc[c, 'med_bps']:.2f} bps | p75 {disp.loc[c, 'p75_bps']:.2f}")
print(f"  COR 75th pct member cap: {np.nanpercentile(corcaps, 75):.2f} bps")
print(f"  default (C0 p75, eval mean): {float(np.nanmean(dflt[H2_MASK])):.2f} bps")
print(f"  const[May, T={T_EXCL}s]: {CONST_POOL:.2f} bps")
print(f"  LOO75 share of members at/below target: {float(np.mean(val.rev_loo75 <= Q_TARGET)):.0%} "
      f"| worst member {val.rev_loo75.max():.1%}")

## Conclusions

1. Coverage: the share of CoW mainnet attempts on assets a Binance based cap can serve directly, by proxy, or by the pegged rule is printed in Section 3; the uncovered tail is what the cluster default exists for.
2. Clustering: DTW k means on the daily volatility paths gives the operative partition; Wasserstein clustering of the full return distributions agrees at the ARI printed in Section 4, so the structure is not an artifact of the chosen representation. Per the agreed production rule, clustering is a one time calibration of group levels and defaults, not a live mechanism: pairs with data, real or synthetic, get their own direct[1h] cap.
3. Dispersion: the per cluster p90 to p10 ratios of member mean caps and the usable or too wide verdicts are in Section 6.
4. Cost of borrowing: the leave one out revert rates in Section 7 quantify what members experience under the 75th percentile cluster cap, with the percentile frontier showing the protection versus looseness trade at 50, 75 and 90.
5. Default: the highest cap cluster's cap, its level in bps, and the revert rates it implies for everything calmer are in Section 8, together with the graduation rule.
6. COR: the correlated layer is rule based; its member caps and the applied floors are in Sections 6 and 7, and the synthetic never contributes where the real listing exists.
7. Synthetic distortion: the real WBETHETH cap versus the synthetic ratio cap, with the asynchronous-legs explanation and the interactive comparison figure, is in Section 5.
8. Benchmark: the constant cap per CoW fee class uses the empirical May calibrated constant at this notebook's window length; the headline block above gives the side by side numbers for the T26 and T14 variants.
9. Limitations: two months is one volatility regime; Binance microstructure is not DEX microstructure; the subset spans clusters but is not a census; cross rates and the pegged bucket are handled by rule, not by statistics.

Recommendation skeleton, to fill after review:

- Coverage: __ percent of CoW attempts reachable directly or by proxy.
- Clusters: k = __, intuitive membership __, DTW vs Wasserstein agreement ARI = __.
- Synthetic pairs: WBETH/ETH synthetic vs real cap difference __ bps; COW/ETH and BABY/KAITO caps __ / __ bps.
- Dispersion verdict per cluster: __.
- Median member revert rate under the LOO 75th percentile cluster cap: __ vs the 8 percent target; worst member __; median under the LOO median cap: __.
- Default cap: __ bps mean, revert rate for a typical major under it __.
- Proposed rule: own direct[1h] cap where a feed exists; cluster 75th percentile cap where features are computable; default equal to the highest cluster cap otherwise; graduation after __ days.